# **Wan2.1-14b GGUF Porn/NSFW Text-to-Video Generator**

---

## High-Quality, NSFW-ready, LoRA-Supported T2V Colab

- This notebook is focused on generating high-quality NSFW and porn videos using the Wan2.1-14b quantized GGUF models with **full LoRA support** out of the box.
- **One or MORE LoRAs** can be loaded and blended for your style! Simply provide names and strengths in the UI below.
- **Stable, memory efficient and easy-to-use** NSFW defaults for dimensions, steps, CFG, and more.
- Full comments and tips on adding further extensions like IPAdapter, ControlNet, etc.

## Why this notebook?
- Huge: Full LoRA support, batch LoRA loading, any LoRA strength
- Best defaults for NSFW (frames, res, cfg, steps)
- Reduce OOM errors on T4 with aggressive memory clearing
- Optionally randomize seed for variety
- Switch Q5/Q6 with a toggle
- Place for your custom hacks/plugins (see code comments)
---

- You can use the free T4 GPU to run this. T4 is slow for large frame counts or high resolution; Q6 is faster/leaner but slightly less quality than Q5.
- **Default video: 8-12 frames at 480x832, reasonable CFG, steps 22.**
- **To avoid OOM, keep frames below 12 at high res; Q6 recommended for 10-16 frames unless on bigger GPU.**
- Set frames=1 for image-only. PNGs are high quality even at 720x1280, but need fewer VRAM.
- LoRAs must be copied to `/content/ComfyUI/models/lora` before (see comments in the code for easy expansion!).

In [ ]:
# @title 🛠️ Environment & Model Download (including LoRA support)

# --- Base Library Installs ---
!pip install torch==2.6.0 torchvision==0.21.0
!pip install -q torchsde einops diffusers accelerate xformers==0.0.29.post2 av imageio

# --- Download base code + custom nodes ---
%cd /content
!git clone https://github.com/Isi-dev/ComfyUI
%cd /content/ComfyUI/custom_nodes
!git clone https://github.com/Isi-dev/ComfyUI_GGUF.git
%cd /content/ComfyUI/custom_nodes/ComfyUI_GGUF
!pip install -r requirements.txt
%cd /content/ComfyUI
!apt -y install -qq aria2 ffmpeg

# --- Q5/Q6 MODEL SWITCH ---
useQ6 = False # @param {"type":"boolean"} # Set True to use Q6 (Faster, less VRAM, slightly less quality)

# Set Model Names (can edit for custom models)
WAN_Q5_MODEL = "wan2.1-t2v-14b-Q5_0.gguf"
WAN_Q6_MODEL = "wan2.1-t2v-14b-Q6_K.gguf"

# Download Unet
unet_target = WAN_Q6_MODEL if useQ6 else WAN_Q5_MODEL
unet_url = f"https://huggingface.co/city96/Wan2.1-T2V-14B-gguf/resolve/main/{unet_target}"
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M "$unet_url" -d /content/ComfyUI/models/unet -o "$unet_target"

# --- Always Download Text Encoder and VAE (shared for Q5/Q6) ---
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://huggingface.co/Comfy-Org/Wan_2.1_ComfyUI_repackaged/resolve/main/split_files/text_encoders/umt5_xxl_fp8_e4m3fn_scaled.safetensors -d /content/ComfyUI/models/text_encoders -o umt5_xxl_fp8_e4m3fn_scaled.safetensors
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://huggingface.co/Comfy-Org/Wan_2.1_ComfyUI_repackaged/resolve/main/split_files/vae/wan_2.1_vae.safetensors -d /content/ComfyUI/models/vae -o wan_2.1_vae.safetensors

# --- (Optional) Download LoRAs below for your favorite NSFW styles ----
# Example: !wget -O /content/ComfyUI/models/lora/myNSFWlora.safetensors https://huggingface.co/path/to/your_lora.safetensors
# You may add as many LoRAs as you like here!

import torch
import numpy as np
from PIL import Image
import gc
import sys, random, os
import imageio
from google.colab import files
from IPython.display import display, HTML, Image as IPImage
sys.path.insert(0, '/content/ComfyUI')

# --- ComfyUI Model/Node Imports ---
from comfy import model_management
from nodes import (
    CheckpointLoaderSimple, CLIPLoader, CLIPTextEncode, VAEDecode, VAELoader, KSampler, UNETLoader
)
from custom_nodes.ComfyUI_GGUF.nodes import UnetLoaderGGUF
from comfy_extras.nodes_model_advanced import ModelSamplingSD3
from comfy_extras.nodes_hunyuan import EmptyHunyuanLatentVideo
from comfy_extras.nodes_images import SaveAnimatedWEBP
from comfy_extras.nodes_video import SaveWEBM

from comfy.nodes_lora import LoraLoader # <-- Built-in LoRA loader

# --- Node Instantiation ---
unet_loader = UnetLoaderGGUF()
# model_sampling = ModelSamplingSD3() # For SDXL/patching, not needed in Wan 2.1
clip_loader = CLIPLoader()
clip_encode_positive = CLIPTextEncode()
clip_encode_negative = CLIPTextEncode()
vae_loader = VAELoader()
empty_latent_video = EmptyHunyuanLatentVideo()
ksampler = KSampler()
vae_decode = VAEDecode()
save_webp = SaveAnimatedWEBP()
save_webm = SaveWEBM()
lora_loader = LoraLoader()

def clear_memory(verbose=False):
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()
    count = 0
    for objname, obj in list(globals().items()):
        try:
            if torch.is_tensor(obj) or (hasattr(obj, "data") and torch.is_tensor(obj.data)):
                del globals()[objname]
                count += 1
        except:
            continue
    if verbose:
        print(f"[clear_memory] Objects cleaned up: {count}")
    gc.collect()

# --- Video/Image Save Utilities ---
def save_as_mp4(images, filename_prefix, fps, output_dir="/content/ComfyUI/output"):
    os.makedirs(output_dir, exist_ok=True)
    output_path = f"{output_dir}/{filename_prefix}.mp4"
    frames = [(img.cpu().numpy() * 255).astype(np.uint8) for img in images]
    with imageio.get_writer(output_path, fps=fps) as writer:
        for frame in frames:
            writer.append_data(frame)
    return output_path

def save_as_webm(images, filename_prefix, fps, codec="vp9", quality=16, output_dir="/content/ComfyUI/output"):
    os.makedirs(output_dir, exist_ok=True)
    output_path = f"{output_dir}/{filename_prefix}.webm"
    frames = [(img.cpu().numpy() * 255).astype(np.uint8) for img in images]
    kwargs = {
        'fps': int(fps), 'quality': int(quality), 'codec': str(codec), 'output_params': ['-crf', str(int(quality))]
    }
    with imageio.get_writer(output_path, format='FFMPEG', mode='I', **kwargs) as writer:
        for frame in frames:
            writer.append_data(frame)
    return output_path

def save_as_image(image, filename_prefix, output_dir="/content/ComfyUI/output"):
    os.makedirs(output_dir, exist_ok=True)
    output_path = f"{output_dir}/{filename_prefix}.png"
    frame = (image.cpu().numpy() * 255).astype(np.uint8)
    Image.fromarray(frame).save(output_path)
    return output_path

def display_video(video_path):
    from base64 import b64encode
    video_data = open(video_path,'rb').read()
    # Detect MIME for embed
    if video_path.lower().endswith('.mp4'): mime_type = "video/mp4"
    elif video_path.lower().endswith('.webm'): mime_type = "video/webm"
    elif video_path.lower().endswith('.webp'): mime_type = "image/webp"
    else: mime_type = "video/mp4"
    data_url = f"data:{mime_type};base64," + b64encode(video_data).decode()
    display(HTML(f"""
    <video width=512 controls autoplay loop><source src="{data_url}" type="{mime_type}"></video>"""))

print("\u2705 Environment Setup Complete!")
# --- CODE EXTENSIONS: To add ControlNet, IPAdapter, Reactor etc, add their repo/clone and models above and import/patch usage in the generate_video() function and the parameter list below. See project wiki/readmes for more info. ---

In [ ]:
# @title 🍑 Generate NSFW Video/Image - Full LoRA, Q5/Q6, and SFW/NSFW Prompts

"""
Tips for best results:
- For fast porn loops: use 8-10 frames, 480x832 or 512x832 (keep <12 for T4)
- Default CFG 1.2, steps 22: more natural, less overdone skin
- Use seed = -1 for FULL randomness every run, else fix seed for reroll
- To load multiple LoRAs: supply comma list in lora_name, and a comma/space-list lora_strength. Example:
    lora_name = 'sexy_butt_v12.safetensors,super_gigaporn_mix1.0.safetensors'
    lora_strength = '0.7, 0.5'
- LoRAs must be present in /content/ComfyUI/models/lora (see cell above for wget)
"""

positive_prompt = "ultra high res, explicit uncensored, stunning beautiful nude female pornstar, erotic pose, perfect skin, photoreal, 8K pop, sweaty, detailed anatomy, seductive, cumshot, realism" # @param {"type":"string"}
negative_prompt = "censored, mosaic, watermark, worst quality, ugly, deformed, jpeg, extra limbs, mutilated, blurry, cropped, jpeg artifacts, out of frame, poorly drawn, signature, artist name, missing finger, extra hands, extra foot, nipples hidden, bad composition, poorly rendered genitals" # @param {"type":"string"}

width = 480 # @param {"type":"integer"}
height = 832 # @param {"type":"integer"}
frames = 10 # @param {"type":"integer", "min":1, "max":32} # Careful above 12 on T4 GPU
fps = 14 # @param {"type":"integer", "min":1, "max":30}

steps = 22 # @param {"type":"integer", "min":10, "max":60}
cfg_scale = 1.2 # @param {"type":"number", "min":0.8, "max":3.0, "step":0.05}
sampler_name = "uni_pc" # @param ["uni_pc", "euler", "dpmpp_2m", "ddim", "lms"]
scheduler = "simple" # @param ["simple", "normal", "karras", "exponential"]

# --- LoRA Parameters ---
lora_name = "" # @param {"type":"string"} # CSV of .safetensors names (in /content/ComfyUI/models/lora/)
lora_strength = "0.7" # @param {"type":"string"} # CSV or Space-separated for multiple LoRAs (must match lora_name count)

output_format = "mp4" # @param ["mp4", "webm"]
useQ6 = False # @param {"type":"boolean"} # Q6 is lighter/faster (set above if you want globally)

seed = -1 # @param {"type":"integer"} # If -1, a random new seed is chosen per run

# Advanced: Feel free to add parameters for ControlNet, Reactor, etc. Add here and pass into the function below!

def generate_video(
    positive_prompt: str, negative_prompt: str,
    width: int, height: int, seed: int, steps: int, cfg_scale: float,
    sampler_name: str, scheduler: str, frames: int, fps: int,
    lora_name: str = "", lora_strength: str = "1.0", useQ6: bool = False,
    output_format: str = "mp4"
):
    """Main generation logic with LoRA support and aggressive memory management."""
    if seed == -1:
        seed = random.randint(0, 2**63 - 1)
        print(f"[INFO] Random seed chosen: {seed}")

    with torch.inference_mode():
        # --- Text Encoder ---
        print("[INFO] Loading Text Encoder...")
        clip = clip_loader.load_clip("umt5_xxl_fp8_e4m3fn_scaled.safetensors", "wan", "default")[0]
        positive = clip_encode_positive.encode(clip, positive_prompt)[0]
        negative = clip_encode_negative.encode(clip, negative_prompt)[0]
        clear_memory()

        # --- Empty Latent ---
        empty_latent = empty_latent_video.generate(width, height, frames, 1)[0]
        clear_memory()

        # --- Load Unet Model (Q5 or Q6 as selected) ---
        print(f"[INFO] Loading Unet Model ({'Q6' if useQ6 else 'Q5'}) ...")
        unet_path = "wan2.1-t2v-14b-Q6_K.gguf" if useQ6 else "wan2.1-t2v-14b-Q5_0.gguf"
        model = unet_loader.load_unet(unet_path)[0]

        # --- LoRA Support: Multiple LoRAs with custom strengths ---
        applied_loras = []
        if lora_name.strip():
            names = [n.strip() for n in lora_name.split(',') if n.strip()]
            strengths = [float(x.strip()) for x in lora_strength.replace(',', ' ').split()]
            if len(strengths) < len(names):
                strengths += [strengths[-1]] * (len(names)-len(strengths))
            print(f"[INFO] Loading {len(names)} LoRA(s): {names} w/ strengths {strengths}")
            for l_name, l_strength in zip(names, strengths):
                lora_path = os.path.join('/content/ComfyUI/models/lora', l_name)
                if not os.path.exists(lora_path):
                    print(f"[WARN] Could not find LoRA: {lora_path}. Skipping.")
                    continue
                # LoRA Loader: returns patched model
                model = lora_loader.load_lora(model, lora_path, l_strength)[0]
                applied_loras.append(l_name)
            if not applied_loras:
                print('[WARN] No LoRA(s) actually loaded!')
        clear_memory()

        # --- Main Sampling ---
        print("[INFO] Sampling video latents...")
        sampled = ksampler.sample(
            model=model,
            seed=seed,
            steps=steps,
            cfg=cfg_scale,
            sampler_name=sampler_name,
            scheduler=scheduler,
            positive=positive,
            negative=negative,
            latent_image=empty_latent
        )[0]
        del model; clear_memory()

        # --- VAE decode ---
        print("[INFO] Loading VAE and decoding latents...")
        vae = vae_loader.load_vae("wan_2.1_vae.safetensors")[0]
        decoded = vae_decode.decode(vae, sampled)[0]
        del vae; clear_memory()

        # --- Save/Display Output ---
        output_path = ""
        if frames == 1:
            print("[INFO] Saving single PNG image...")
            output_path = save_as_image(decoded[0], "ComfyUI")
            display(IPImage(filename=output_path))
        else:
            if output_format.lower() == "webm":
                print("[INFO] Saving as WEBM...")
                output_path = save_as_webm(
                    decoded, "ComfyUI", fps=fps, codec="vp9")
            elif output_format.lower() == "mp4":
                print("[INFO] Saving as MP4...")
                output_path = save_as_mp4(decoded, "ComfyUI", fps)
            else:
                raise ValueError(f"Unsupported output format: {output_format}")
            display_video(output_path)

        # Always clean up as much RAM as possible after run
        clear_memory(verbose=True)

# --- MAIN TRIGGER: Call generation function ---
generate_video(
    positive_prompt=positive_prompt,
    negative_prompt=negative_prompt,
    width=width,
    height=height,
    seed=seed,
    steps=steps,
    cfg_scale=cfg_scale,
    sampler_name=sampler_name,
    scheduler=scheduler,
    frames=frames,
    fps=fps,
    lora_name=lora_name,
    lora_strength=lora_strength,
    useQ6=useQ6,
    output_format=output_format
)
# --- Final memory cleanup for next run ---
clear_memory(verbose=True)

# ---
# [EXTENSION COMMENT]:
# To add features like ControlNet, Reactor, IPAdapter, DreamBooth finetune, etc, simply:
#   - Install their code/repos at the top cell
#   - Download their weights
#   - Pass extra params in generate_video() and call in the main function accordingly.
#   - For inspiration see networks like ComfyUI IPAdapter node or custom nodes for ControlNet/Adapter/Segment-Anything.
#   - WAN 2.1 is patchable, so you can even add SD3-style adapters in the same vein! ---